# Demostración visual del capítulo 4

Este cuaderno sirve para enseñar al tutor el recorrido completo del piloto de pimiento en Almería. No realiza llamadas a las APIs: trabaja con evidencias ya guardadas para que la reunión no dependa de tokens, conexión ni cuotas.

**Orden:** datos originales -> normalización -> tabla de modelización -> análisis exploratorio -> evaluación -> pruebas.

## 1 Localizar el repositorio y cargar las librerías

La celda funciona tanto si Visual Studio Code abre el notebook desde la raíz como desde la carpeta `notebooks`.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'docs').exists():
    ROOT = ROOT.parent

print('Repositorio:', ROOT)
print('Evidencias:', ROOT / 'docs' / 'data_samples')

Repositorio: C:\Users\adria\OneDrive\Documentos\ChatGPT\TFM\tfm-regadio-dss
Evidencias: C:\Users\adria\OneDrive\Documentos\ChatGPT\TFM\tfm-regadio-dss\docs\data_samples


## 2 Qué llegó desde SiAR

El archivo bruto conserva la consulta, la fecha de ingestión, el número de registros y los datos tal como llegaron. Ejemplo real del histórico AL01:

```json
{
  "source": "SiAR",
  "dataset": "weather_daily",
  "query": {
    "station": "AL01",
    "start_date": "2025-05-01",
    "end_date": "2025-09-30",
    "chunk_count": 2
  },
  "record_count": 150
}
```

El JSON completo está localmente en `data/raw/siar/weather/daily/station=AL01/period=2025-05-01_2025-09-30`.

## 3 Cómo se consulta la API SiAR

El cliente lee `SIAR_API_TOKEN` desde `.env`, construye los parámetros y nunca imprime la credencial. [Abrir siar_client.py en GitHub](https://github.com/al1sr/tfm-regadio-dss/blob/main/src/data/siar_client.py).

In [2]:
# Muestra el método real sin llamar a la API ni leer el token.
source = (ROOT / 'src/data/siar_client.py').read_text(encoding='utf-8')
start = source.index('    def get_data(')
end = source.index('    @staticmethod', start)
print(source[start:end])

    def get_data(
        self,
        data_type: str,
        scope: str,
        identifiers: str | list[str],
        start_date: str | date,
        end_date: str | date,
        *,
        calculated: bool = False,
        last_modified: str | date | None = None,
    ) -> list[dict[str, Any]]:
        """Consulta series agroclimáticas por estación, provincia o comunidad.

        ``identifiers`` puede contener uno o varios códigos obtenidos mediante
        los catálogos del endpoint ``Info``.
        """
        data_types = {
            "HORARIOS": "Horarios",
            "DIARIOS": "Diarios",
            "SEMANALES": "Semanales",
            "MENSUALES": "Mensuales",
        }
        scopes = {"CCAA", "PROVINCIA", "ESTACION"}
        normalized_type = data_type.upper()
        normalized_scope = scope.upper()
        if normalized_type not in data_types:
            raise ValueError(
                "Tipo de datos no válido. Valores admitidos: "
                + ", ".join(d

## 4 Necesidades hídricas del pimiento

Este CSV aporta la referencia agronómica. La variable objetivo es `net_irrigation_need_mm`, equivalente a `ETc - Pe`.

In [3]:
needs = pd.read_csv(ROOT / 'docs/data_samples/siar_necesidades_pimiento_AL01_2025_sample.csv')
needs.head(6)

,date,crop_coefficient_kc,et0_mm,crop_evapotranspiration_etc_mm,effective_precipitation_mm,net_irrigation_need_mm,quality_status
0,2025-05-01,0.5,4.72,2.36,0.00,2.36,PASS
1,2025-05-14,0.5,2.20,1.10,0.17,0.93,PASS
2,2025-05-23,0.5,3.51,1.75,2.34,0.00,PASS
3,2025-06-01,0.7,6.08,4.25,0.00,4.25,PASS
4,2025-07-01,0.9,6.05,5.45,0.00,5.45,PASS
5,2025-07-21,0.9,NaN,NaN,NaN,NaN,WARN


| date | Kc | ET0 mm | ETc mm | Pe mm | necesidad neta mm | calidad |
|---|---:|---:|---:|---:|---:|---|
| 2025-05-01 | 0,50 | 4,72 | 2,36 | 0,00 | 2,36 | PASS |
| 2025-05-14 | 0,50 | 2,20 | 1,10 | 0,17 | 0,93 | PASS |
| 2025-05-23 | 0,50 | 3,51 | 1,75 | 2,34 | 0,00 | PASS |
| 2025-06-01 | 0,70 | 6,08 | 4,25 | 0,00 | 4,25 | PASS |
| 2025-07-21 | 0,90 | - | - | - | - | WARN |

[Abrir el CSV completo en GitHub](https://github.com/al1sr/tfm-regadio-dss/blob/main/data/external/siar_necesidades_pimiento_AL01_2025.csv)

## 5 Cómo se integra AEMET

AEMET devuelve primero una respuesta de control con una URL temporal y después se descarga el producto. [Abrir aemet_client.py en GitHub](https://github.com/al1sr/tfm-regadio-dss/blob/main/src/data/aemet_client.py).

In [4]:
# Muestra la descarga en dos pasos sin llamar a AEMET.
source = (ROOT / 'src/data/aemet_client.py').read_text(encoding='utf-8')
start = source.index('    def _get_product(')
end = source.index('    def get_municipalities', start)
print(source[start:end])

    def _get_product(self, endpoint: str) -> dict[str, Any] | list[Any]:
        control = self._request_json(
            f"{self._base_url}/{endpoint.lstrip('/')}", authenticated=True
        )
        if not isinstance(control, dict):
            raise AEMETError("La respuesta de control de AEMET no es válida.")

        data_url = control.get("datos")
        if not isinstance(data_url, str) or not data_url:
            description = control.get("descripcion") or "sin URL de descarga"
            raise AEMETError(f"AEMET no proporcionó datos: {description}")

        self._validate_download_url(data_url)
        return self._request_json(data_url, authenticated=False)




**Situación actual:** AEMET está conectado y normalizado, pero no se usa en este entrenamiento porque la predicción disponible es de 2026 y el ciclo SiAR es de 2025.

## 6 Del dato bruto a la tabla de modelización

![Flujo de procesamiento](../docs/images/procesamiento_datos_4_1.png)

| etapa | conjunto | filas | variables | filas válidas | avisos |
|---|---|---:|---:|---:|---:|
| raw | SiAR meteorológico | 150 | JSON original | n.a. | n.a. |
| raw externo | necesidades del pimiento | 150 | 6 | n.a. | n.a. |
| interim | SiAR normalizado | 150 | 22 | 148 | 2 |
| interim | necesidades normalizadas | 150 | 15 | 148 | 2 |
| processed | tabla analítica integrada | 150 | 88 | 148 etiquetas | 2 |

La limpieza mantiene las 150 filas para conservar trazabilidad, pero solo 148 tienen objetivo válido. No hay duplicados y existen tres huecos de calendario. [Abrir auditoría reproducible](https://github.com/al1sr/tfm-regadio-dss/blob/main/docs/data_samples/etl_volume_summary_4_1.csv).

In [5]:
modeling = pd.read_csv(ROOT / 'docs/data_samples/modeling_dataset_sample_4_2.csv')
columns_to_show = [
    'observed_date', 'temperature_mean_c', 'humidity_mean_pct',
    'crop_coefficient_kc', 'target_net_irrigation_need_mm',
    'net_irrigation_need_mm_lag_1d', 'quality_status'
]
modeling[columns_to_show].head()

,observed_date,temperature_mean_c,humidity_mean_pct,crop_coefficient_kc,target_net_irrigation_need_mm,net_irrigation_need_mm_lag_1d,quality_status
0,2025-05-01,19.62,75.60,0.5,2.36,NaN,PASS
1,2025-05-02,21.47,59.55,0.5,2.78,2.36,PASS
2,2025-05-03,19.93,58.28,0.5,2.63,2.78,PASS
3,2025-05-04,18.17,70.10,0.5,2.46,2.63,PASS
4,2025-05-05,18.03,72.00,0.5,2.42,2.46,PASS


| fecha | temperatura | humedad | Kc | objetivo | objetivo día anterior | calidad |
|---|---:|---:|---:|---:|---:|---|
| 2025-05-01 | 19,62 | 75,60 | 0,50 | 2,36 | - | PASS |
| 2025-05-02 | 21,47 | 59,55 | 0,50 | 2,78 | 2,36 | PASS |
| 2025-05-03 | 19,93 | 58,28 | 0,50 | 2,63 | 2,78 | PASS |
| 2025-05-04 | 18,17 | 70,10 | 0,50 | 2,46 | 2,63 | PASS |
| 2025-05-05 | 18,03 | 72,00 | 0,50 | 2,42 | 2,46 | PASS |

[Abrir muestra](https://github.com/al1sr/tfm-regadio-dss/blob/main/docs/data_samples/modeling_dataset_sample_4_2.csv) · [Abrir catálogo de 88 variables](https://github.com/al1sr/tfm-regadio-dss/blob/main/docs/data_samples/feature_catalog_4_2.csv)

## 7 Análisis exploratorio

```python
summary = json.loads((ROOT / 'docs/data_samples/eda_summary_4_2.json').read_text(encoding='utf-8'))
summary['scope'], summary['dataset'], summary['target']
```

- 150 días observados y 3 fechas ausentes.
- 148 valores válidos de la variable objetivo.
- 0 duplicados y 0 necesidades negativas.
- Necesidad neta media de 3,588 mm por día.

![Serie temporal](../docs/images/eda_series_4_2.png)

![Calidad](../docs/images/eda_quality_4_2.png)

## 8 Modelos y evaluación temporal

El desarrollo usa 125 filas y la prueba final 23 filas, siempre respetando el orden temporal. Se conservan persistencia, Ridge y Random Forest y se añaden KNN, SVR y XGBoost.

In [ ]:
report = json.loads((ROOT / 'docs/data_samples/model_evaluation_4_3.json').read_text(encoding='utf-8'))
pd.DataFrame({
    model: {
        'MAE validación': report['cross_validation']['summary'][model]['mae_mm_day']['mean'],
        'MAE prueba': report['test'][model]['mae_mm_day'],
    }
    for model in ['persistence', 'ridge', 'random_forest', 'knn', 'svr', 'xgboost']
}).T

| enfoque | MAE validación | MAE prueba |
|---|---:|---:|
| persistencia | 0,434 | **0,120** |
| Ridge | 0,602 | 0,314 |
| Random Forest | 1,118 | 0,562 |
| KNN | 0,937 | 0,544 |
| SVR | 1,052 | 1,486 |
| XGBoost | 1,141 | 0,766 |

La persistencia no es un algoritmo de aprendizaje, sino el baseline temporal: para estimar el día t usa la necesidad observada en t−1. Se mantiene como referencia técnica porque ningún modelo ML la mejora de forma estable con esta muestra. Ridge queda como el mejor candidato de machine learning.

![Validación](../docs/images/model_validation_4_3.png)

![Predicciones](../docs/images/model_predictions_4_3.png)

## 9 Pruebas automáticas

```powershell
python -m unittest discover -s tests -p "test_*.py" -v
```

Resultado verificado: `Ran 40 tests` y `OK`. Estas pruebas no consumen cuota de las APIs. [Abrir tests en GitHub](https://github.com/al1sr/tfm-regadio-dss/tree/main/tests).

## 10 Preguntas que debemos resolver con el tutor

1. ¿Mantenemos el alcance nacional y usamos Almería pimiento como piloto?
2. ¿Predecimos directamente `ETc - Pe` o usamos ML para corregir el cálculo agronómico?
3. ¿Cuántas campañas, estaciones o cultivos debemos incorporar?
4. ¿Qué producto final espera: estudio, API, dashboard o aplicación DSS?
5. ¿Debemos incorporar sensores y riego realmente aplicado?
6. ¿Mantenemos MySQL y Apache Hop en la arquitectura final?